<a href="https://colab.research.google.com/github/keerthireddy-23/FUTURE_ML_02/blob/main/Support_Ticket_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install nltk scikit-learn

import pandas as pd
import numpy as np
import re
import nltk


In [2]:
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [4]:
from google.colab import files
uploaded = files.upload()


Saving customer_support_tickets.csv to customer_support_tickets.csv


In [25]:
df = pd.read_csv("customer_support_tickets.csv")

print(df.head())
print(df.columns)


   Ticket ID        Customer Name              Customer Email  Customer Age  \
0          1        Marisa Obrien  carrollallison@example.com            32   
1          2         Jessica Rios    clarkeashley@example.com            42   
2          3  Christopher Robbins   gonzalestracy@example.com            48   
3          4     Christina Dillon    bradleyolson@example.org            27   
4          5    Alexander Carroll     bradleymark@example.com            67   

  Customer Gender Product Purchased Date of Purchase      Ticket Type  \
0           Other        GoPro Hero       2021-03-22  Technical issue   
1          Female       LG Smart TV       2021-05-22  Technical issue   
2           Other          Dell XPS       2020-07-14  Technical issue   
3          Female  Microsoft Office       2020-11-13  Billing inquiry   
4          Female  Autodesk AutoCAD       2020-02-04  Billing inquiry   

             Ticket Subject  \
0             Product setup   
1  Peripheral compatibil

In [26]:
df['combined_text'] = (
    df['Ticket Description'].astype(str) + " " +
    df['Ticket Subject'].astype(str) + " " +
    df['Ticket Type'].astype(str)
)


In [27]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_text'] = df['combined_text'].apply(clean_text)


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=7000, ngram_range=(1,2))

X = vectorizer.fit_transform(df['clean_text'])


In [29]:
from sklearn.model_selection import train_test_split

y_category = df['Ticket Type']

X_train, X_test, y_train, y_test = train_test_split(
    X, y_category, test_size=0.2, random_state=42
)


In [30]:
from sklearn.naive_bayes import MultinomialNB

category_model = MultinomialNB()
category_model.fit(X_train, y_train)


MultinomialNB()

In [31]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = category_model.predict(X_test)

print("Category Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Category Accuracy: 0.9557260920897285

Classification Report:
                       precision    recall  f1-score   support

     Billing inquiry       0.96      0.97      0.97       357
Cancellation request       0.96      0.93      0.94       327
     Product inquiry       0.92      0.91      0.91       316
      Refund request       0.96      0.98      0.97       345
     Technical issue       0.99      0.98      0.98       349

            accuracy                           0.96      1694
           macro avg       0.95      0.95      0.95      1694
        weighted avg       0.96      0.96      0.96      1694



In [33]:
y_priority = df['Ticket Priority']

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X, y_priority, test_size=0.2, random_state=42
)


In [34]:
priority_model = MultinomialNB()
priority_model.fit(X_train_p, y_train_p)


MultinomialNB()

In [35]:
y_pred_p = priority_model.predict(X_test_p)

print("Priority Accuracy:", accuracy_score(y_test_p, y_pred_p))
print("\nPriority Report:\n", classification_report(y_test_p, y_pred_p))


Priority Accuracy: 0.24911452184179456

Priority Report:
               precision    recall  f1-score   support

    Critical       0.24      0.30      0.26       411
        High       0.24      0.24      0.24       409
         Low       0.23      0.18      0.20       415
      Medium       0.28      0.27      0.28       459

    accuracy                           0.25      1694
   macro avg       0.25      0.25      0.25      1694
weighted avg       0.25      0.25      0.25      1694



In [42]:
def final_priority(text, category):
    text = text.lower()

    if "urgent" in text or "immediately" in text:
        return "Critical"

    if category == "Technical issue":
        return "High"

    if "failed" in text or "error" in text:
        return "High"

    return "Medium"


In [43]:
def predict_ticket(text):
    clean = clean_text(text)
    vec = vectorizer.transform([clean])

    category = category_model.predict(vec)[0]
    priority = final_priority(text, category)

    print("\nTicket:", text)
    print("Category:", category)
    print("Priority:", priority)


In [44]:
predict_ticket("My payment failed and I need urgent help")
predict_ticket("I cannot login to my account")



Ticket: My payment failed and I need urgent help
Category: Product inquiry
Priority: Critical

Ticket: I cannot login to my account
Category: Refund request
Priority: Medium


In [45]:
from sklearn.metrics import confusion_matrix

print("Category Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nPriority Confusion Matrix:\n", confusion_matrix(y_test_p, y_pred_p))


Category Confusion Matrix:
 [[348   1   8   0   0]
 [  0 304  13   9   1]
 [ 15   5 287   5   4]
 [  0   6   2 337   0]
 [  0   2   3   1 343]]

Priority Confusion Matrix:
 [[122 102  86 101]
 [123 100  82 104]
 [121 106  75 113]
 [145 112  77 125]]
